In [0]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, max as spark_max,
    min as spark_min, round as spark_round, when, lit,
    current_timestamp, desc, countDistinct
)

In [0]:
# configuracion de rutas


storage_account_name = "stfrauddatabricks"

container_silver = "silver"
container_gold = "gold"

project_folder = "fraud_detection"

silver_base_path = f"abfss://{container_silver}@{storage_account_name}.dfs.core.windows.net/{project_folder}"
gold_base_path = f"abfss://{container_gold}@{storage_account_name}.dfs.core.windows.net/{project_folder}"

print("SILVER path:", silver_base_path)
print("GOLD path  :", gold_base_path)

In [0]:
# Ruta Silver de entrada 
silver_transactions_path = f"{silver_base_path}/transactions_clean_enriched_delta"
silver_transaction_type_risk_path = f"{silver_base_path}/dim_transaction_type_risk_delta"
silver_customer_profile_path = f"{silver_base_path}/dim_customer_account_profile_delta"

print("Silver transacciones :", silver_transactions_path)
print("Silver riesgo tipo   :", silver_transaction_type_risk_path)
print("Silver perfiles      :", silver_customer_profile_path)

In [0]:
#Ruta gold dalida
gold_kpi_fraud_summary_path = f"{gold_base_path}/gold_kpi_fraud_summary_delta"
gold_fraud_by_transaction_type_path = f"{gold_base_path}/gold_fraud_by_transaction_type_delta"
gold_fraud_by_risk_level_path = f"{gold_base_path}/gold_fraud_by_risk_level_delta"
gold_fraud_by_amount_category_path = f"{gold_base_path}/gold_fraud_by_amount_category_delta"
gold_fraud_by_profile_risk_path = f"{gold_base_path}/gold_fraud_by_profile_risk_delta"
gold_top_suspicious_accounts_path = f"{gold_base_path}/gold_top_suspicious_accounts_delta"
gold_fraud_by_step_path = f"{gold_base_path}/gold_fraud_by_step_delta"
gold_dashboard_executive_summary_path = f"{gold_base_path}/gold_dashboard_executive_summary_delta"

print("Rutas Gold configuradas correctamente.")

In [0]:
#Leer datos de silver 
df_transactions_silver = spark.read.format("delta").load(silver_transactions_path)
df_transaction_type_risk_silver = spark.read.format("delta").load(silver_transaction_type_risk_path)
df_customer_profile_silver = spark.read.format("delta").load(silver_customer_profile_path)

print("Registros Silver:")
print("Transacciones :", df_transactions_silver.count())
print("Riesgo tipo   :", df_transaction_type_risk_silver.count())
print("Perfiles      :", df_customer_profile_silver.count())

display(df_transactions_silver.limit(10))

In [0]:
# Tabla Gold: KPIs generales de fraude

df_gold_kpi_fraud_summary = (
    df_transactions_silver
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("total_fraud_transactions"),
        spark_sum("is_flagged_fraud").alias("total_flagged_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_transaction_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("total_fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_transaction_amount"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_silver_risk_score"),
        countDistinct("origin_account_id").alias("unique_origin_accounts"),
        countDistinct("destination_account_id").alias("unique_destination_accounts")
    )
    .withColumn(
        "fraud_transaction_rate_pct",
        spark_round((col("total_fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn(
        "fraud_amount_rate_pct",
        spark_round((col("total_fraud_amount") / col("total_transaction_amount")) * 100, 4)
    )
    .withColumn("gold_updated_at", current_timestamp())
)

display(df_gold_kpi_fraud_summary)

In [0]:
#Tabla Gold: Fraude por tipo de transacción

# COMMAND ----------

df_gold_fraud_by_transaction_type = (
    df_transactions_silver
    .groupBy("transaction_type", "risk_level")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("fraud_transactions"),
        spark_sum("is_flagged_fraud").alias("flagged_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_amount"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_silver_risk_score")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn("gold_updated_at", current_timestamp())
    .orderBy(desc("fraud_transactions"))
)

display(df_gold_fraud_by_transaction_type)

In [0]:
# Tabla Gold: Fraude por nivel de riesgo Silver

# COMMAND ----------

df_gold_fraud_by_risk_level = (
    df_transactions_silver
    .groupBy("silver_fraud_risk_level")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("fraud_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("fraud_amount"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_silver_risk_score"),
        spark_max("silver_fraud_risk_score").alias("max_silver_risk_score"),
        spark_min("silver_fraud_risk_score").alias("min_silver_risk_score")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn("gold_updated_at", current_timestamp())
    .orderBy(desc("avg_silver_risk_score"))
)

display(df_gold_fraud_by_risk_level)

In [0]:
# Tabla Gold: Fraude por categoría de monto
# COMMAND ----------

df_gold_fraud_by_amount_category = (
    df_transactions_silver
    .groupBy("amount_category")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("fraud_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_amount"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_silver_risk_score")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn("gold_updated_at", current_timestamp())
    .orderBy(desc("fraud_amount"))
)

display(df_gold_fraud_by_amount_category)

In [0]:

# Tabla Gold: Fraude por perfil de riesgo


df_gold_fraud_by_profile_risk = (
    df_transactions_silver
    .groupBy(
        "origin_account_type",
        "origin_customer_segment",
        "origin_kyc_status",
        "origin_is_high_risk_profile",
        "profile_match_flag"
    )
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("fraud_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("fraud_amount"),
        spark_round(avg("origin_device_risk_score"), 2).alias("avg_origin_device_risk_score"),
        spark_round(avg("origin_login_failed_attempts"), 2).alias("avg_origin_login_failed_attempts"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_silver_risk_score")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn("gold_updated_at", current_timestamp())
    .orderBy(desc("fraud_transactions"))
)

display(df_gold_fraud_by_profile_risk)

In [0]:
# Ranking de cuentas sospechosas
# COMMAND ----------

df_gold_top_suspicious_accounts = (
    df_transactions_silver
    .groupBy(
        "origin_account_id",
        "origin_account_type",
        "origin_customer_segment",
        "origin_kyc_status",
        "origin_country",
        "origin_region",
        "origin_device_type",
        "origin_is_high_risk_profile"
    )
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("fraud_transactions"),
        spark_sum("is_flagged_fraud").alias("flagged_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_amount"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_silver_risk_score"),
        spark_max("silver_fraud_risk_score").alias("max_silver_risk_score")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn(
        "suspicious_account_score",
        spark_round(
            (col("fraud_transactions") * 30) +
            (col("flagged_transactions") * 20) +
            (col("fraud_rate_pct") * 2) +
            (col("avg_silver_risk_score") * 1.5),
            2
        )
    )
    .withColumn("gold_updated_at", current_timestamp())
    .orderBy(desc("suspicious_account_score"))
)

display(df_gold_top_suspicious_accounts.limit(20))

In [0]:
# Fraude por step


df_gold_fraud_by_step = (
    df_transactions_silver
    .groupBy("step")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("fraud_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("fraud_amount"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_silver_risk_score")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn("gold_updated_at", current_timestamp())
    .orderBy("step")
)

display(df_gold_fraud_by_step.limit(20))

In [0]:
# Resumen ejecutivo para dashboard

df_gold_dashboard_executive_summary = (
    df_transactions_silver
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("is_fraud").alias("fraud_transactions"),
        spark_sum("is_flagged_fraud").alias("flagged_transactions"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == 1, col("amount")).otherwise(lit(0))), 2).alias("fraud_amount"),
        spark_round(avg("silver_fraud_risk_score"), 2).alias("avg_risk_score"),
        spark_sum(when(col("silver_fraud_risk_level") == "CRITICAL", 1).otherwise(0)).alias("critical_risk_transactions"),
        spark_sum(when(col("silver_fraud_risk_level") == "HIGH", 1).otherwise(0)).alias("high_risk_transactions"),
        spark_sum(when(col("profile_match_flag") == 1, 1).otherwise(0)).alias("transactions_with_profile_match")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round((col("fraud_transactions") / col("total_transactions")) * 100, 4)
    )
    .withColumn(
        "flagged_vs_real_fraud_pct",
        spark_round((col("flagged_transactions") / col("fraud_transactions")) * 100, 4)
    )
    .withColumn(
        "profile_match_rate_pct",
        spark_round((col("transactions_with_profile_match") / col("total_transactions")) * 100, 4)
    )
    .withColumn("gold_updated_at", current_timestamp())
)

display(df_gold_dashboard_executive_summary)

In [0]:
# Guardar tablas Gold en Delta


(
    df_gold_kpi_fraud_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_kpi_fraud_summary_path)
)

(
    df_gold_fraud_by_transaction_type.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_fraud_by_transaction_type_path)
)

(
    df_gold_fraud_by_risk_level.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_fraud_by_risk_level_path)
)

(
    df_gold_fraud_by_amount_category.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_fraud_by_amount_category_path)
)

(
    df_gold_fraud_by_profile_risk.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_fraud_by_profile_risk_path)
)

(
    df_gold_top_suspicious_accounts.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_top_suspicious_accounts_path)
)

(
    df_gold_fraud_by_step.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_fraud_by_step_path)
)

(
    df_gold_dashboard_executive_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_dashboard_executive_summary_path)
)

print("Tablas Gold guardadas correctamente en formato Delta.")

In [0]:
# Validar lectura desde Gold

df_gold_kpi_check = spark.read.format("delta").load(gold_kpi_fraud_summary_path)
df_gold_type_check = spark.read.format("delta").load(gold_fraud_by_transaction_type_path)
df_gold_risk_check = spark.read.format("delta").load(gold_fraud_by_risk_level_path)
df_gold_amount_check = spark.read.format("delta").load(gold_fraud_by_amount_category_path)
df_gold_profile_check = spark.read.format("delta").load(gold_fraud_by_profile_risk_path)
df_gold_accounts_check = spark.read.format("delta").load(gold_top_suspicious_accounts_path)
df_gold_step_check = spark.read.format("delta").load(gold_fraud_by_step_path)
df_gold_summary_check = spark.read.format("delta").load(gold_dashboard_executive_summary_path)

print("Validación de tablas Gold:")
print("KPI fraude              :", df_gold_kpi_check.count())
print("Fraude por tipo         :", df_gold_type_check.count())
print("Fraude por riesgo       :", df_gold_risk_check.count())
print("Fraude por monto        :", df_gold_amount_check.count())
print("Fraude por perfil       :", df_gold_profile_check.count())
print("Top cuentas sospechosas :", df_gold_accounts_check.count())
print("Fraude por step         :", df_gold_step_check.count())
print("Resumen ejecutivo       :", df_gold_summary_check.count())